In [ ]:
import os
import re
import json
import vtk 
import slicer   
import MultiVolumeImporter  
import ExtractCenterline 

############################################################################
############################################################################

### SETUP/IMPORT ### 

#===========================================================================================
# CHANGE FILE PATHS, SEGMENT NAMES, PREPROCESSING VALUES AND OTHER MODIFIERS AS NEEDED
#===========================================================================================

#---------------------------------------------------------------------------------------------------
# PATH TO CASE LABEL FILE
#filepath = r"Z:\public_resources\RA4-Project\D-Images\MESAL-3\2-Checked-Done-Temp\Done\Case-20-MESAL-7015569-E3\Markups\88-Jensen\S-Final-4-label.nrrd"
#---------------------------------------------------------------------------------------------------

# GET CASE NUMBER SUBSTRING FROM FILE PATH
#caseNum = re.search(r'Case-\d{2}', filepath, re.IGNORECASE).group()
caseNum = '22'

print("Loading " + caseNum + " label file for centerline extraction...")

# LOAD FILE AS VOLUME (CURRENTLY SAME AS SEGMENTATION) 
volume = slicer.util.loadVolume(filepath)

# LOAD IN A SEGMENTATION FILE 
# USED TO SPECIFY THAT FUTURE PROCESSES WILL INTERACT WITH THIS FILE (NOTE FUTURE USES OF 'vessel_seg')
vessel_seg = slicer.util.loadSegmentation(filepath)

print(caseNum + " label file loaded...")

segmentation = vessel_seg.GetSegmentation() 

# OPTION TO APPLY MODIFIER TO LOADED FILE (SMOOTHING EXAMPLE) 
segmentation.SetConversionParameter("Smoothing factor", "0.1") 

# OPTION TO CREATE THE 3D VIEW 
vessel_seg.CreateClosedSurfaceRepresentation() 

# ASSIGNS SPECIFIC SEGMENTATION LABEL ('Segment_1') TO VARIABLE 
segmentID = segmentation.GetSegmentIdBySegmentName('Segment_1') 

# CREATES AN EMPTY 'vtkPolyData' OBJECT 
segmentVtkPolyData = vtk.vtkPolyData() 

# GET 'vtkPolyData' REPRESENTATION OF SEGMENT SURFACE INTO 'segmentVtkPolyData' VARIABLE 
    # BY PASSING IN SEGMENTATION LABEL & 'vtkPolyData' VARIABLES 
vessel_seg.GetClosedSurfaceRepresentation(segmentID, segmentVtkPolyData) 

print(segmentID + " loaded for centerline extraction...") 


In [ ]:
############################################################################
############################################################################

### PREPROCESSING ### 
# USED TO SHORTN PATHS TO FUNCTIONS USED IN MODULE 
extractLogic = ExtractCenterline.ExtractCenterlineLogic() 

## PREPROCESS SEGMENTATION LABLEL ('Segment_1' IN THIS CASE) 
# USED TO INITIALIZE 'EXTRACT CENTERLINE' PARAMETERS 
inputSurfacePolyData = extractLogic.polyDataFromNode(vessel_seg, segmentID) 

# CAN BE MODIFIED AS NEEDED 
# SEE 'ExtractCenterline.py' FOR DEFAULT SETTINGS 
preprocessEnabled = True  # (self._parameterNode.GetParameter("PreprocessInputSurface") == "true") 
targetNumberOfPoints = 15000.0  # float(self._parameterNode.GetParameter("TargetNumberOfPoints")) 
decimationAggressiveness = 4.0 #float(self._parameterNode.GetParameter("DecimationAggressiveness")) 
subdivideInputSurface = False  # (self._parameterNode.GetParameter("SubdivideInputSurface") == "true") 
autoDetectEndPointsPushButton = True # (self._parameterNode.GetParameter(autoDetectEndPointsPushButton") == "true") 

slicer.util.showStatusMessage( 
    "Preprocessing surface before centerline extraction..." 
) 

print("Preprocessing surface before centerline extraction...") 

slicer.app.processEvents() 

# PASSESES IN THE ABOVE PREPROCESS VALUES 
# WILL BE USED DURING CENTERLINE EXTRACTION 
preprocessedPolyData = extractLogic.preprocess( 
    inputSurfacePolyData, 
    targetNumberOfPoints, 
    decimationAggressiveness, 
    subdivideInputSurface, 
) 
  
print("Preprocessing complete...")

In [ ]:
############################################################################
############################################################################

### EXTRACT CENTERLINE ### 

centerlineCurveNode = None 

print("Auto-detecting surface model endpoints...") 

# ATTEMPTS TO 'CLICK' THE 'AUTO-DETECT' BUTTON AND STORE THE END POINT MARKUPS IN THE 'endPointsMarkupsNode' VARIABLE 
# Auto-detect the endpoints 
endPointsMarkupsNode = slicer.mrmlScene.AddNewNodeByClass("vtkMRMLMarkupsFiducialNode", "Centerline endpoints") 
networkPolyData = extractLogic.extractNetwork(preprocessedPolyData, endPointsMarkupsNode) 
startPointPosition=None 
endpointPositions = extractLogic.getEndPoints(networkPolyData, startPointPosition) 
endPointsMarkupsNode.RemoveAllMarkups() 

for position in endpointPositions: 
  endPointsMarkupsNode.AddControlPoint(vtk.vtkVector3d(position)) 

print("End points autodetected...")

# USED TO STORE THE CENTERLINE CURVE
if centerlineCurveNode is None: 
    centerlineCurveNode = slicer.mrmlScene.AddNewNodeByClass("vtkMRMLMarkupsCurveNode", "Centerline curve") 

slicer.util.showStatusMessage("Extracting centerline...")  

# FORCES UPDATE TO ENSURE MODIFIED SETTINGS ARE APPLIED BEFORE RUNNING THE MODULE 
slicer.app.processEvents()  # force update 

# PASSES CENTERLINE/DIAGRAM DATA TO 2 SEPARATE VARIABLES 
# 'preprocessedPolyData'PREPROCESS MODIFIERS, MARKUPS/ENDPOINTS NODE PASSED INTO 'extractCenterline' 
centerlinePolyData, voronoiDiagramPolyData = extractLogic.extractCenterline( 
        preprocessedPolyData, endPointsMarkupsNode, curveSamplingDistance=0.001) #'end_pts' replaced with 'endPointsMarkupsNode' 

slicer.util.showStatusMessage("Generating curves and quantification results table...") 

slicer.app.processEvents()  # force update  

centerlinePropertiesTableNode = None  

print("Extracting centerline curve tree...")

# USED TO GENERATE CENTERLINE SEGMENTS FROM DATA (INDIVIDUAL AIRWAY GENERATION SEGMENTS)
# extractLogic.createCurveTreeFromCenterline( 
#     centerlinePolyData, centerlineCurveNode, centerlinePropertiesTableNode 
# )

if centerlineCurveNode.GetNumberOfControlPoints() == 2: 
    # If Extraction had an error, likely due to too high decimation aggressiveness 
    # Try again 
    slicer.util.errorDisplay( 
        "Centerline generation failed, possibly due to decimationAggressiveness being too high." 
    )

print("Segmented centerline curve tree created...")

In [ ]:
############################################################################
############################################################################

### GET 1-SEGMENT CENTERLINE MODEL ###

print("Extracting centerline model...")

# INSTANTIATE MODEL NODE
centerlineModel = slicer.vtkMRMLModelNode()
# APPLY 'centerlinePolyData'TO NEW MODEL NODE
centerlineModel.SetAndObservePolyData(centerlinePolyData)
# USED TO EDIT CENTERLINE MODEL APPEARANCE
centerlineModelDisplay = slicer.vtkMRMLModelDisplayNode()

# USED TO SET APPEARANCE OF MODEL NODE
centerlineModelDisplay.SetColor(1, 1, 0)
centerlineModelDisplay.BackfaceCullingOff()
centerlineModelDisplay.SetOpacity(0.8)
centerlineModelDisplay.SetPointSize(3)

centerlineModelDisplay.SetSliceIntersectionVisibility(True)
centerlineModelDisplay.SetVisibility(True)

# ADDS MODEL NODE APPEARANCE INFO TO SLICER SCENE
slicer.mrmlScene.AddNode(centerlineModelDisplay)
centerlineModel.SetAndObserveDisplayNodeID(centerlineModelDisplay.GetID())
centerlineModelDisplay.SetInputPolyDataConnection(centerlineModel.GetPolyDataConnection())
# ADDS CENTERLINE MODEL NODE TO SLICER SCENE
slicer.mrmlScene.AddNode(centerlineModel)

print("Centerline extraction complete...")

In [ ]:
############################################################################
############################################################################

### EXPORT RESULTS TO FILES ###

print("Exporting 1-segment centerline model to file...")

outfile = "complete_centerline_model.vtk"
dirPath = os.path.dirname(filepath)
outPath = os.path.join(dirPath, outfile)

# EXPORT 1-SEGMENT CENTERLINE MODEL TO .vtk FILE
slicer.util.saveNode(centerlineModel, outPath)

print(caseNum + " 1-segment centerline model exported to:")
print(outPath)

In [ ]:
# MULTI-CASE 1-SEGMENT CENTERLINE EXTRACTION

import os
import re
import shutil
import json
import vtk 
import slicer   
import MultiVolumeImporter  
import ExtractCenterline
import time

def find_and_copy_highest_numbered_file(root_dir):

    start_time = time.time()

    copied_filepaths = []  # List to store copied file paths
    
    # Loop through each case directory
    for case_dir in os.listdir(root_dir):
        case_dir_path = os.path.join(root_dir, case_dir)
        
        # Check if the item in the root directory is indeed a directory,
        # not named "Archive", and does not start with "Case-32" (case insensitive)
        if (os.path.isdir(case_dir_path) and 
            not re.match(r'archive', case_dir, re.IGNORECASE) and
            not case_dir.lower().startswith("1-Duplicates") and
            not case_dir.lower().startswith("case-32")): # case-32 excluded due to centerline extraction issues
            
            # Traverse subdirectories to find a directory beginning with "88-"
            for subdir, dirs, _ in os.walk(case_dir_path):
                if subdir.split(os.sep)[-1].startswith("88-"):
                    if "complete_centerline_curves_model.vtk" in os.listdir(subdir):
                        caseNum = re.search(r'(?i)case-\d+', subdir).group()
                        print(f"Skipping {caseNum} - 'complete_centerline_curves_model.vtk' file found in {subdir}")
                        break  # Skip this case directory if the file exists
                    else:
                        max_num = float('-inf')
                        max_filename = None
                    # max_num = float('-inf')
                    # max_filename = None
                    
                    # Loop through files in the found directory
                    for file in os.listdir(subdir):
                        # Check if file matches the pattern
                        if re.match(r'S-Final-(\d+)-label\.nrrd', file):
                            match = re.match(r'S-Final-(\d+)-label\.nrrd', file)
                            num = int(match.group(1))
                            if num > max_num:
                                max_num = num
                                max_filename = file
                                
                    if max_filename:
                        src_file = os.path.join(subdir, max_filename)
                        dest_file = os.path.join(subdir, max_filename.replace("-label.nrrd", "-label-Copy.nrrd"))
                        caseNum = re.search(r'(?i)case-\d+', src_file).group()
                        print(caseNum)
                        # Check if source file exists before copying
                        if os.path.exists(src_file):
                            try:
                                shutil.copy(src_file, dest_file)
                                copied_filepaths.append(dest_file)
                                # Now you can use 'copied_filepaths' list in other functions
                                print(f"Copied {src_file} to {dest_file}")
                            except IOError as e:
                                print(f"Error copying file: {e}")
                        else:
                            print(f"Source file not found: {src_file}")
                    break  # Once a directory starting with "88-" is found, break the loop
    
    end_time = time.time()
    elapsed_time = end_time - start_time
    # Convert elapsed time to minutes and seconds
    minutes = int(elapsed_time // 60)
    seconds = int(elapsed_time % 60)
    if(minutes == 1):
        print("Elapsed time: {} minute and {} seconds".format(minutes, seconds))
    else:
        print("Elapsed time: {} minutes and {} seconds".format(minutes, seconds))

    print("----------------------------------------------------------------------\n")

    return copied_filepaths

def centerline_curve(file):

    start_time = time.time()

    #print("Loading file for centerline extraction...")

    filepath = file

    caseNum = re.search(r'case-\d{2}', filepath, re.IGNORECASE).group()

    print(caseNum + " - extracting centerline curve...")

    # LOAD FILE AS VOLUME (CURRENTLY SAME AS SEGMENTATION) 
    volume = slicer.util.loadVolume(filepath)

    # LOAD IN A SEGMENTATION FILE 
    # USED TO SPECIFY THAT FUTURE PROCESSES WILL INTERACT WITH THIS FILE (NOTE FUTURE USES OF 'vessel_seg')
    vessel_seg = slicer.util.loadSegmentation(filepath)

    segmentation = vessel_seg.GetSegmentation() 

    # OPTION TO APPLY MODIFIER TO LOADED FILE (SMOOTHING EXAMPLE) 
    segmentation.SetConversionParameter("Smoothing factor", "0.1") 

    # OPTION TO CREATE THE 3D VIEW 
    vessel_seg.CreateClosedSurfaceRepresentation() 

    # ASSIGNS SPECIFIC SEGMENTATION LABEL ('Segment_1') TO VARIABLE 
    segmentID = segmentation.GetSegmentIdBySegmentName('Segment_1') 

    # CREATES AN EMPTY 'vtkPolyData' OBJECT 
    segmentVtkPolyData = vtk.vtkPolyData() 

    # GET 'vtkPolyData' REPRESENTATION OF SEGMENT SURFACE INTO 'segmentVtkPolyData' VARIABLE 
        # BY PASSING IN SEGMENTATION LABEL & 'vtkPolyData' VARIABLES 
    vessel_seg.GetClosedSurfaceRepresentation(segmentID, segmentVtkPolyData) 

    #print(segmentID + " loaded for centerline extraction...") 

    ############################################################################
    ############################################################################

    ### PREPROCESSING ### 
    # USED TO SHORTN PATHS TO FUNCTIONS USED IN MODULE 
    extractLogic = ExtractCenterline.ExtractCenterlineLogic() 

    ## PREPROCESS SEGMENTATION LABLEL ('Segment_1' IN THIS CASE) 
    # USED TO INITIALIZE 'EXTRACT CENTERLINE' PARAMETERS 
    inputSurfacePolyData = extractLogic.polyDataFromNode(vessel_seg, segmentID) 

    # CAN BE MODIFIED AS NEEDED 
    # SEE 'ExtractCenterline.py' FOR DEFAULT SETTINGS 
    preprocessEnabled = True  # (self._parameterNode.GetParameter("PreprocessInputSurface") == "true") 
    targetNumberOfPoints = 15000.0  # float(self._parameterNode.GetParameter("TargetNumberOfPoints")) 
    decimationAggressiveness = 4.0 #float(self._parameterNode.GetParameter("DecimationAggressiveness")) 
    subdivideInputSurface = False  # (self._parameterNode.GetParameter("SubdivideInputSurface") == "true") 
    autoDetectEndPointsPushButton = True # (self._parameterNode.GetParameter(autoDetectEndPointsPushButton") == "true") 

    slicer.util.showStatusMessage( 
        "Preprocessing surface before centerline extraction..." 
    ) 

    #print("Preprocessing surface before centerline extraction...") 

    slicer.app.processEvents() 

    # PASSESES IN THE ABOVE PREPROCESS VALUES 
    # WILL BE USED DURING CENTERLINE EXTRACTION 
    preprocessedPolyData = extractLogic.preprocess( 
        inputSurfacePolyData, 
        targetNumberOfPoints, 
        decimationAggressiveness, 
        subdivideInputSurface,
    ) 
    
    #print("Preprocessing complete...")

    ############################################################################
    ############################################################################

    ### EXTRACT CENTERLINE ### 

    centerlineCurveNode = None 

    #print("Auto-detecting surface model endpoints...") 

    # ATTEMPTS TO 'CLICK' THE 'AUTO-DETECT' BUTTON AND STORE THE END POINT MARKUPS IN THE 'endPointsMarkupsNode' VARIABLE 
    # Auto-detect the endpoints 
    endPointsMarkupsNode = slicer.mrmlScene.AddNewNodeByClass("vtkMRMLMarkupsFiducialNode", "Centerline endpoints")
    networkPolyData = extractLogic.extractNetwork(preprocessedPolyData, endPointsMarkupsNode) 
    startPointPosition=None 
    endpointPositions = extractLogic.getEndPoints(networkPolyData, startPointPosition)
    endPointsMarkupsNode.RemoveAllMarkups()

    for position in endpointPositions: 
        endPointsMarkupsNode.AddControlPoint(vtk.vtkVector3d(position)) 

    #print("End points autodetected...")

    # USED TO STORE THE CENTERLINE CURVE
    if centerlineCurveNode is None: 
        centerlineCurveNode = slicer.mrmlScene.AddNewNodeByClass("vtkMRMLMarkupsCurveNode", "Centerline curve") 

    slicer.util.showStatusMessage("Extracting centerline...")
    #print("Extracting centerline...")  

    # FORCES UPDATE TO ENSURE MODIFIED SETTINGS ARE APPLIED BEFORE RUNNING THE MODULE 
    slicer.app.processEvents()  # force update 

    # PASSES CENTERLINE/DIAGRAM DATA TO 2 SEPARATE VARIABLES 
    # 'preprocessedPolyData'PREPROCESS MODIFIERS, MARKUPS/ENDPOINTS NODE PASSED INTO 'extractCenterline' 
    centerlinePolyData, voronoiDiagramPolyData = extractLogic.extractCenterline( 
            preprocessedPolyData, endPointsMarkupsNode, curveSamplingDistance=0.001) #'end_pts' replaced with 'endPointsMarkupsNode' 

    #print(centerlinePolyData)

    slicer.util.showStatusMessage("Generating curves and quantification results table...")
    #print("Generating curves and quantification results table...")

    slicer.app.processEvents()  # force update  

    centerlinePropertiesTableNode = None  

    #print("Extracting centerline curve tree...")

    # USED TO GENERATE CENTERLINE SEGMENTS FROM DATA
    # extractLogic.createCurveTreeFromCenterline( 
    #     centerlinePolyData, centerlineCurveNode, centerlinePropertiesTableNode 
    # )

    if centerlineCurveNode.GetNumberOfControlPoints() == 2: 
        # If Extraction had an error, likely due to too high decimation aggressiveness 
        # Try again 
        slicer.util.errorDisplay( 
            "Centerline generation failed, possibly due to decimationAggressiveness being too high." 
        )

    #print("Segmented centerline curve tree created...")

    ############################################################################
    ############################################################################

    ### GET 1-SEGMENT CENTERLINE MODEL ###

    #print("Extracting centerline model...")

    # INSTANTIATE MODEL NODE
    centerlineModel = slicer.vtkMRMLModelNode()
    # APPLY 'centerlinePolyData'TO NEW MODEL NODE
    centerlineModel.SetAndObservePolyData(centerlinePolyData)
    # USED TO EDIT CENTERLINE MODEL APPEARANCE
    centerlineModelDisplay = slicer.vtkMRMLModelDisplayNode()

    # USED TO SET APPEARANCE OF MODEL NODE
    centerlineModelDisplay.SetColor(1, 1, 0)
    centerlineModelDisplay.BackfaceCullingOff()
    centerlineModelDisplay.SetOpacity(0.8)
    centerlineModelDisplay.SetPointSize(3)

    centerlineModelDisplay.SetSliceIntersectionVisibility(True)
    centerlineModelDisplay.SetVisibility(True)

    # ADDS MODEL NODE APPEARANCE INFO TO SLICER SCENE
    slicer.mrmlScene.AddNode(centerlineModelDisplay)
    centerlineModel.SetAndObserveDisplayNodeID(centerlineModelDisplay.GetID())
    centerlineModelDisplay.SetInputPolyDataConnection(centerlineModel.GetPolyDataConnection())
    # ADDS CENTERLINE MODEL NODE TO SLICER SCENE
    slicer.mrmlScene.AddNode(centerlineModel)

    #print("Centerline model extracted")

    #print("Centerline extraction complete")

    ############################################################################
    ############################################################################

    ### EXPORT 1-SEGMENT CENTERLINE TO FILE ###

    #print("Exporting 1-segment centerline model to file...")

    outfile = "complete_centerline_curves_model.vtk"
    dirPath = os.path.dirname(filepath)
    outPath = os.path.join(dirPath, outfile)

    # EXPORT 1-SEGMENT CENTERLINE MODEL TO .vtk FILE
    slicer.util.saveNode(centerlineModel, outPath)

    end_time = time.time()
    elapsed_time = end_time - start_time
    # Convert elapsed time to minutes and seconds
    minutes = int(elapsed_time // 60)
    seconds = int(elapsed_time % 60)

    print(caseNum + " 1-segment centerline curves model exported to:")
    print(outPath)
    if(minutes == 1):
        print("Elapsed time: {} minute and {} seconds".format(minutes, seconds))
    else:
        print("Elapsed time: {} minutes and {} seconds".format(minutes, seconds))
    print("======================================================================\n")


##########################################################################################
##########################################################################################
##########################################################################################


#root_directory = r"Z:\public_resources\RA4-Project\D-Images\SPIROMICS-SubStudy\2-Results-CheckedDoneTemp\1-Done\Case-32-Spiromics-51292112"
root_directory = r"Z:\public_resources\RA4-Project\D-Images\SPIROMICS-SubStudy\2-Results-CheckedDoneTemp\1-Done"
copied_files = find_and_copy_highest_numbered_file(root_directory)
for file in copied_files:
    centerline_curve(file)